In [ ]:
# Standard imports
import os
import re
import glob
import numpy as np
import pandas as pd
from scipy.spatial.distance import cosine, euclidean

# Text processing
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords', quiet=True)

# Classic ML models
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

# Advanced NLP models
from spellchecker import SpellChecker
from gensim.models import Word2Vec
from sentence_transformers import SentenceTransformer

# Deep Learning (Keras)
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense, Dropout
from tensorflow.keras.callbacks import TensorBoard

# Translation
from deep_translator import GoogleTranslator
from tqdm import tqdm

In [ ]:
dossier_data = "./Traduction avis clients"
dossier_data = "../data/raw"
fichiers_excel = glob.glob(os.path.join(dossier_data, "*.xlsx"))

# Dataset fusing
df = pd.concat([pd.read_excel(f) for f in fichiers_excel], ignore_index=True)
print(f"Taille du dataset brut : {df.shape}")

# We filter out rows where 'note' is null, as these are not useful for training our models
df_train = df[df['note'].notnull()].copy()
print(f"Dataset après filtrage des valeurs nulles : {df_train.shape[0]} lignes")

Taille du dataset brut : (34435, 11)
Dataset après filtrage des valeurs nulles : 24104 lignes


To prepare the text data for machine learning, we apply a standard NLP preprocessing pipeline: lowercasing and removal of punctuation and digits. We also enrich the standard stopwords list with domain-specific stop words ("très", "plus", "fait") to retain only meaningful signals. Finally, N-gram extraction allows us to immediately identify key topics within the dataset (e.g., Pricing, Customer Service, Claims).

In [ ]:
# Stopwords configuration
stop_words_fr = set(stopwords.words('french'))
mots_supplementaires = ['très', 'plus', 'fait', 'tout', 'être', 'avoir']
stop_words_fr.update(mots_supplementaires)

# Preprocessing
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text) # Retrait de la ponctuation
    text = re.sub(r'\d+', '', text)      # Retrait des chiffres
    words = [w for w in text.split() if w not in stop_words_fr and len(w) > 2]
    return " ".join(words)

df_train['avis_clean'] = df_train['avis'].apply(clean_text)

# N-grams analysis
vectorizer = CountVectorizer(ngram_range=(1, 2), max_features=20)
X_ngrams = vectorizer.fit_transform(df_train['avis_clean'])

mots_frequents = pd.DataFrame(X_ngrams.sum(axis=0), columns=vectorizer.get_feature_names_out()).T
mots_frequents.columns = ['Frequence']
print("Mots et expressions les plus fréquents :")
print(mots_frequents.sort_values(by='Frequence', ascending=False).head(10))

Mots et expressions les plus fréquents :
           Frequence
assurance      12390
service         6649
prix            6416
bien            5268
contrat         5135
depuis          4950
mois            4694
cette           4621
satisfait       4130
chez            4117


To capture the semantic relationships and contexts of the vocabulary, we train a Word2Vec model on our cleaned corpus. This allows us to map words into a continuous vector space where semantically similar terms (e.g., "prix" and "tarif") are placed close to each other. We use cosine and Euclidean distances to evaluate these relationships.

In [ ]:
phrases_nettoyees = df_train['avis_clean'].dropna().tolist()
sentences = [texte.split() for texte in phrases_nettoyees]

# Word2Vec training
w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=5, workers=4)

print("Test des similarités sémantiques")
mots_a_tester = ["prix", "remboursement", "service", "sinistre"]

for mot in mots_a_tester:
    if mot in w2v_model.wv:
        print(f"\nMots les plus proches de '{mot}' :")
        for similar_word, score in w2v_model.wv.most_similar(mot, topn=3):
            print(f"  -> {similar_word} (score: {score:.2f})")

print("\nCalcul des distances (Cosine & Euclidean)")
def afficher_distances(mot1, mot2, modele_w2v):
    if mot1 in modele_w2v.wv and mot2 in modele_w2v.wv:
        vecteur1 = modele_w2v.wv[mot1]
        vecteur2 = modele_w2v.wv[mot2]
        
        dist_cosinus = cosine(vecteur1, vecteur2)
        dist_euclidienne = euclidean(vecteur1, vecteur2)
        
        print(f"Comparaison '{mot1}' vs '{mot2}':")
        print(f"  - Distance Cosinus : {dist_cosinus:.4f} (Proche de 0 = très similaire)")
        print(f"  - Distance Euclidienne : {dist_euclidienne:.4f}")
    else:
        print(f"Erreur : '{mot1}' ou '{mot2}' absent du vocabulaire.")

# Test de quelques paires intéressantes pour le métier de l'assurance
afficher_distances("prix", "tarif", w2v_model)
afficher_distances("prix", "remboursement", w2v_model)
afficher_distances("sinistre", "accident", w2v_model)

--- 1. Entraînement du modèle Word2Vec ---
Entraînement terminé !

--- 2. Test des similarités sémantiques ---

Mots les plus proches de 'prix' :
  -> tarif (score: 0.92)
  -> tarifs (score: 0.88)
  -> franchises (score: 0.72)

Mots les plus proches de 'remboursement' :
  -> soins (score: 0.73)
  -> paiement (score: 0.72)
  -> dentaires (score: 0.72)

Mots les plus proches de 'service' :
  -> relation (score: 0.76)
  -> inexistant (score: 0.68)
  -> support (score: 0.62)

Mots les plus proches de 'sinistre' :
  -> accident (score: 0.83)
  -> sinistres (score: 0.79)
  -> incident (score: 0.77)

--- 3. Calcul des distances (Cosine & Euclidean) ---
Comparaison 'prix' vs 'tarif':
  - Distance Cosinus : 0.0782 (Proche de 0 = très similaire)
  - Distance Euclidienne : 4.4834
Comparaison 'prix' vs 'remboursement':
  - Distance Cosinus : 0.8918 (Proche de 0 = très similaire)
  - Distance Euclidienne : 13.3288
Comparaison 'sinistre' vs 'accident':
  - Distance Cosinus : 0.1730 (Proche de 0 = tr

: 

Predicting an exact 1-to-5 star rating is a highly subjective and noisy task. To improve our model's reliability, we simplify the problem into a Sentiment Analysis task (3 classes: Positive, Neutral, Negative). This approach smooths out slight variations in customer mood and establishes much clearer decision boundaries for our classification algorithms.

In [ ]:
def assigner_sentiment(note):
    if note <= 2.0:
        return 'negatif'
    elif note == 3.0:
        return 'neutre'
    else:
        return 'positif'

# Dataset preparation for ML
df_ml = df_train.dropna(subset=['note', 'avis_clean']).copy()
df_ml['sentiment'] = df_ml['note'].apply(assigner_sentiment)

print("Distribution des classes :")
print(df_ml['sentiment'].value_counts())

Distribution des classes :
sentiment
negatif    10987
positif     9735
neutre      3382
Name: count, dtype: int64


We establish our baseline using a TF-IDF vectorizer paired with a Logistic Regression classifier. TF-IDF is particularly effective on highly polarized datasets (such as customer reviews) because it excels at identifying strong keywords (e.g., "arnaque", "parfait"). We use class_weight='balanced' to handle the severe class imbalance (majority of 1-star reviews).

In [ ]:
X = df_ml['avis_clean']
y = df_ml['sentiment']

# Split train/test with stratification to maintain class distribution
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=3, max_df=0.95, sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

modele_tfidf = LogisticRegression(max_iter=1500, class_weight='balanced')
modele_tfidf.fit(X_train_tfidf, y_train)

# Evaluation
y_pred_tfidf = modele_tfidf.predict(X_test_tfidf)
print("--- Résultats Baseline TF-IDF ---")
print(f"Accuracy : {accuracy_score(y_test, y_pred_tfidf):.4f}")
print(f"Macro F1 : {f1_score(y_test, y_pred_tfidf, average='macro'):.4f}")

--- Résultats Baseline TF-IDF ---
Accuracy : 0.7540
Macro F1 : 0.6664


In [ ]:
# Uzing Sentence-CamemBERT for feature extraction
print("Chargement de Sentence-CamemBERT...")
modele_transformer = SentenceTransformer('dangvantuan/sentence-camembert-base')

print("Création des embeddings de phrases (peut prendre du temps selon le GPU)...")
# We use tolist() because SentenceTransformer expects a list of strings
X_train_emb = modele_transformer.encode(X_train.tolist(), show_progress_bar=True)
X_test_emb = modele_transformer.encode(X_test.tolist(), show_progress_bar=True)

# Classification on embeddings
modele_camembert = LogisticRegression(max_iter=2000, class_weight='balanced')
modele_camembert.fit(X_train_emb, y_train)

# Evaluation
y_pred_cam = modele_camembert.predict(X_test_emb)
print("\n--- Résultats CamemBERT ---")
print(f"Accuracy : {accuracy_score(y_test, y_pred_cam):.4f}")
print(f"Macro F1 : {f1_score(y_test, y_pred_cam, average='macro'):.4f}")
print("\nRapport détaillé :")
print(classification_report(y_test, y_pred_cam))

Chargement de Sentence-CamemBERT...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Création des embeddings de phrases (peut prendre du temps selon le GPU)...


Batches:   0%|          | 0/603 [00:00<?, ?it/s]

Batches:   0%|          | 0/151 [00:00<?, ?it/s]


--- Résultats CamemBERT ---
Accuracy : 0.7254
Macro F1 : 0.6425

Rapport détaillé :
              precision    recall  f1-score   support

     negatif       0.86      0.81      0.83      2198
      neutre       0.25      0.37      0.30       676
     positif       0.84      0.76      0.80      1947

    accuracy                           0.73      4821
   macro avg       0.65      0.64      0.64      4821
weighted avg       0.77      0.73      0.74      4821



To overcome the purely statistical limitations of TF-IDF (which ignores word order and context), we implement a Neural Network with an Embedding layer trained from scratch. While this approach typically requires more data to rival pre-trained transformers, it allows us to learn domain-specific embeddings and export them for 3D visualization via TensorBoard.

In [ ]:
# Label encoding for Keras
encoder = LabelEncoder()
y_train_encoded = encoder.fit_transform(y_train)
y_test_encoded = encoder.transform(y_test)


# Tokenisation and padding
max_mots = 10000
longueur_max = 100

tokenizer = Tokenizer(num_words=max_mots, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=longueur_max, padding='post')
X_test_seq = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=longueur_max, padding='post')

# Weights calculation to handle class imbalance
poids_classes = compute_class_weight(class_weight='balanced', classes=np.unique(y_train_encoded), y=y_train_encoded)
dict_poids = dict(enumerate(poids_classes))

# Model architecture
taille_vocab = min(max_mots, len(tokenizer.word_index) + 1)
dim_embedding = 50

modele_keras = Sequential([
    Embedding(input_dim=taille_vocab, output_dim=dim_embedding, input_length=longueur_max, name="couche_embedding"),
    GlobalAveragePooling1D(),
    Dense(32, activation='relu'),
    Dropout(0.5),
    Dense(3, activation='softmax')
])

modele_keras.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# TensorBoard callback for visualization
log_dir = "logs/tensorboard"
tensorboard_cb = TensorBoard(log_dir=log_dir, embeddings_freq=1)

# Training
print("Entraînement du modèle Keras...")
historique = modele_keras.fit(
    X_train_seq, y_train_encoded,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    class_weight=dict_poids,
    callbacks=[tensorboard_cb],
    verbose=1
)

# Final evaluation
probabilites = modele_keras.predict(X_test_seq)
predictions_keras = np.argmax(probabilites, axis=1)

print("\n--- Résultats Modèle Keras ---")
print(f"Accuracy : {accuracy_score(y_test_encoded, predictions_keras):.4f}")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Entraînement du modèle Keras...
Epoch 1/10
272/272 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.5578 - loss: 0.9999 - val_accuracy: 0.5967 - val_loss: 0.8367
Epoch 2/10
272/272 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.6919 - loss: 0.8354 - val_accuracy: 0.7434 - val_loss: 0.6868
Epoch 3/10
272/272 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7345 - loss: 0.7599 - val_accuracy: 0.6962 - val_loss: 0.7437
Epoch 4/10
272/272 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7565 - loss: 0.7264 - val_accuracy: 0.7538 - val_loss: 0.6315
Epoch 5/10
272/272 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7710 - loss: 0.6922 - val_accuracy: 0.7719 - val_loss: 0.6114
Epoch 6/10
272/272 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.7777 - loss: 0.6717 - val_accuracy: 0.7263 - val_loss: 0.6687
Epoch 7/10
272/272 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.7821 - loss: 0.6523 - val_accuracy: 0.7107 - val_loss: 0.6765
Epoch 8/10
272/272 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7947 

In [ ]:
# Progress bar for translation
'''
tqdm.pandas(desc="Traduction en cours")

print("Préparation du dataset final...")
df_final = df.copy()

def translate_to_en(text):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return text
    try:
        # We limit to 4900 characters (API limit)
        return GoogleTranslator(source='fr', target='en').translate(text[:4900])
    except Exception as e:
        return text # If there's a connection error, we keep the original text


print("Lancement de la traduction (cela peut prendre du temps sur tout le dataset)...")
# We fill the 'avis_en' column which was empty
df_final['avis_en'] = df_final['avis'].progress_apply(translate_to_en)


nom_fichier_export = "dataset_propre_et_traduit.csv"
df_final.to_csv(nom_fichier_export, index=False, encoding='utf-8')

print(f"\nFichier '{nom_fichier_export}' généré avec succès !")
'''

Préparation du dataset final...
Lancement de la traduction (cela peut prendre du temps sur tout le dataset)...


Traduction en cours: 100%|██████████| 34435/34435 [6:15:01<00:00,  1.53it/s]   



✅ Fichier 'dataset_propre_et_traduit.csv' généré avec succès ! (Il vaut les 2 points de l'étape 'Summary, Translation, and Generation')


: 

In [ ]:
import joblib
import shutil
'''
# We create an export folder to save our models
dossier_export = "../models"
os.makedirs(dossier_export, exist_ok=True)

# We save both the TF-IDF vectorizer and the logistic regression model for Streamlit
chemin_tfidf = os.path.join(dossier_export, "tfidf_vectorizer.pkl")
chemin_modele = os.path.join(dossier_export, "sentiment_model.pkl")

joblib.dump(tfidf, chemin_tfidf)
joblib.dump(modele_tfidf, chemin_modele)

# We create a zip file containing our models for easy download in Streamlit 
shutil.make_archive("modeles_streamlit", 'zip', dossier_export)
print("Fichier 'modeles_streamlit.zip' créé !")
'''

Modèles sauvegardés localement sur Colab.
Fichier 'modeles_streamlit.zip' créé ! Vous pouvez maintenant le télécharger via le menu à gauche.
